# Embeddings & Vector Stores

**Module 8 · Lesson 8.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Embedding APIs - Text to Vectors
- ChromaDB - Local Vector Store for Prototyping
- AlloyDB pgvector - SQL + Vectors in One Database
- Vertex AI Vector Search - Billion-Scale Production
- MTEB Rankings - Commercial vs Open-Source Embedding Models
- Matryoshka Embeddings - Variable Dimensions, Two-Stage Retrieval

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy chromadb python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Million-Vector Morning

> 💡 **Info**
>
> Your 8.1 prototype stored 40 vectors in a Python list and answered in milliseconds. Six months later the corpus is 4 million chunks. Every query now compares against every vector: two full seconds of numpy per question, the API bill for re-embedding grows weekly, and the CEO asks why the German office gets wrong answers (their documents embed fine - retrieval just never finds them). Nothing is BROKEN. Everything is *undersized*: the wrong embedding model, no index, no storage plan. This lesson is the sizing conversation: which embedding API, which vector home, which index - and when each answer changes.

### 🎯 What you will be able to do after this lesson

- **Build** the same retrieval stack on three homes - ChromaDB (prototype), AlloyDB pgvector (SQL + vectors), Vertex AI Vector Search (billion-scale) - and know the migration triggers between them.

- **Compare** embedding models with MTEB as a starting point and your own corpus as the verdict - commercial APIs vs Qwen3/BGE-M3 open source.

- **Evaluate** index structures (HNSW, IVF-PQ, ScaNN, DiskANN) by the currency each spends - RAM, recall, or latency - and tune M/ef_search deliberately.

- **Design** cost-efficient retrieval with Matryoshka truncation, two-stage search, metadata pre-filtering, and the right distance metric.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1-8.2 (embeddings, cosine, chunking, batched `embed_content`), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai numpy chromadb`. Steps 3-4 are read-along (they need cloud instances); everything else runs locally.

## 60-Second Warm-Up: What You Already Know

Three recalls - all load-bearing today. Click a card to check yourself.

> 🗺 **Analogy**
>
> **A vector database is a GPS for text.** Every document gets coordinates (embedding vector). When you ask a question, the GPS finds the nearest documents - not by keyword match, but by *meaning*. "Return policy" and "how to get my money back" are at the same coordinates even though they share no words. **At 10M documents, numpy is a bicycle. A vector DB is a Formula 1 car.**

> 💡 **Info**
>
> ⚠️ Misconception: "pick the #1 MTEB model and the best-funded database"
> Engineers treat embedding models and vector databases like phone upgrades - grab the top benchmark score and the biggest brand, done. That is the anti-pattern to avoid, twice over. Leaderboard rank is measured on MTEB's tasks, not your corpus: a model two points "worse" can beat the leader on YOUR legal documents or Hindi support tickets. And database choice is a SCALING decision, not a status symbol - a dedicated cluster for 200K vectors buys you ops burden with zero benefit, while a Python list at 4M vectors buys you two-second queries. **Rule: benchmark on your data; move homes when a measured limit bites.**

## Build One: The Same Stack on Three Vector Homes

Steps 1-4 are hands-on: embed properly with the current API, then park the SAME vectors in three homes - an embedded library, Postgres, and a billion-scale managed service - so the differences are experienced, not recited.

---

## 🎯 Concept 1: Embedding APIs - Text to Vectors

### Embedding APIs - Text to Vectors

gemini-embedding-001 (batched, task_type, Matryoshka dims) vs open-source. Quality, speed, cost.

**Why start here:** every vector store downstream is only as good as the vectors you feed it. This step locks in the three habits that survive every later migration: batch the call, declare the task_type, and choose dimensions deliberately (Matryoshka truncation). Get these right once and steps 2-4 are just changing the parking spot.

> 🏭 **Analogy**
>
> **An embedding API is a passport office.** Every document gets a passport (vector) that says where it lives in meaning-space. Batch processing matters - one van delivering 500 applications beats 500 separate trips. And the office issues DIFFERENT passport types on purpose: a document passport (RETRIEVAL_DOCUMENT) and a traveler visa (RETRIEVAL_QUERY) are stamped differently so border control (similarity search) matches them correctly.
> **Where the analogy breaks down:** passports are permanent; embeddings are model-versioned. Re-embed everything when you change models - two models' vectors share no common space, and mixing them in one index quietly breaks retrieval.

You are about to embed 2,000 chunks and then thousands of user queries over time. What varies between those two kinds of calls?

**📝 `01_embeddings.py`** - *Gemini*

In [ ]:
# EMBEDDING APIs - Gemini vs open-source
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np, time

client = genai.Client()  # reads GOOGLE_API_KEY from the environment
EMBED = "gemini-embedding-001"

def embed_batch(texts, task="RETRIEVAL_DOCUMENT", dims=768):
    """ONE call for the whole list. task_type matters; dims uses Matryoshka truncation."""
    r = client.models.embed_content(
        model=EMBED, contents=texts,
        config=types.EmbedContentConfig(task_type=task, output_dimensionality=dims))
    return [np.array(e.values) for e in r.embeddings]

# Open-source alternative (local, free, private):
# from sentence_transformers import SentenceTransformer
# local = SentenceTransformer("BAAI/bge-m3"); local.encode(texts)

texts = ["machine learning algorithms", "AI and deep learning models",
         "cooking biryani recipe", "refund and return policy"]

start = time.time()
embs = embed_batch(texts)
elapsed = (time.time() - start) * 1000
print(f"{EMBED}: {len(embs)} texts, {len(embs[0])} dims, {elapsed:.0f}ms (one batched call)\n")

print("Similarity matrix:")
for i, t1 in enumerate(texts):
    row = f"  {t1[:24]:24s}"
    for j in range(len(texts)):
        sim = np.dot(embs[i], embs[j]) / (np.linalg.norm(embs[i]) * np.linalg.norm(embs[j]))
        row += f" {sim:.2f}"
    print(row)

# Output:
#   gemini-embedding-001: 4 texts, 768 dims, 240ms (one batched call)
#   Similarity matrix:
#   machine learning algorith  1.00 0.78 0.31 0.35
#   AI and deep learning mode  0.78 1.00 0.30 0.33
#   cooking biryani recipe     0.31 0.30 1.00 0.28
#   refund and return policy   0.35 0.33 0.28 1.00
#   ML and AI: high similarity. ML and biryani: low. That is semantic search.

# The 2026 menu (prices are per MILLION tokens - check current rates):
#   gemini-embedding-001   3072 dims (Matryoshka)  about $0.15/M   API, MTEB leader
#   OpenAI 3-small         1536 dims               about $0.02/M   API, cheap default
#   OpenAI 3-large         3072 dims               about $0.13/M   API
#   Qwen3-Embedding-8B     4096 dims               Free       Local, Apache 2.0
#   BGE-M3                  1024 dims               Free       Local, 100+ languages

#### 💡 What just happened

⚡ What just happened? Gemini produced 768-dim embeddings. "ML algorithms" and "AI deep learning" scored high similarity (~0.8). "ML" and "biryani" scored low (~0.2). **Key decision: Gemini embeddings for production (fast, cheap API). Local models (MiniLM, BGE) for privacy-sensitive data or offline use. Commercial embedding APIs run about $0.02-0.15 per million tokens depending on the model - budget is rarely the deciding factor at prototype scale.**

---

## 🎯 Concept 2: ChromaDB - Local Vector Store for Prototyping

### ChromaDB - Local Vector Store for Prototyping

pip install, 5 lines to index, 3 lines to search. Perfect for development.

**The upgrade this step buys:** 8.1's Python list forgot everything on restart and checked every vector per query. Chroma is the smallest step up that fixes both - an embedded database (lives inside your Python process, no server) that adds an ANN index, metadata filtering, and optional persistence, while staying a two-line install.

> 📦 **Analogy**
>
> **Chroma is a filing cabinet in your own office.** No building permit, no landlord, no commute - you walk over and file things (pip install, zero infrastructure). It has real drawers and labels (index + metadata), unlike the pile of papers on your desk (the Python list). For one team's worth of documents it is everything you need.
> **Where the analogy breaks down:** a cabinet serves whoever walks up to it - your app process. When three services need the same vectors, or the corpus outgrows one machine's RAM, you need a shared building (client-server: pgvector, Qdrant, Vertex) - the subject of the next two steps.

Your 8.1 Python list now holds 4 million vectors and queries take 2 seconds. What fixes it?

**📝 `02_chromadb.py`** - *ChromaDB*

In [ ]:
# CHROMADB - Local vector database
# pip install chromadb
import chromadb

client = chromadb.Client()  # in-memory; chromadb.PersistentClient(path="./db") to keep it
collection = client.create_collection("netsetos_docs")

# Chroma auto-embeds with its DEFAULT local model (all-MiniLM) - fine for a demo.
# In production pass your own embeddings so index and query use the SAME model.
docs = [
    "Netsetos offers GenAI courses in Hyderabad. 14 modules, 146 hours.",
    "Refund policy: full refund within 7 days. 50% from 8-30 days.",
    "GenAI course costs 14999 rupees. Includes Discord, live sessions, certificate.",
    "Live classes daily at 7 PM IST on YouTube. 85% completion rate.",
    "Prerequisites: basic Python and high school math. No ML experience needed.",
]

collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))],
    metadatas=[{"source": f"faq_{i}.txt"} for i in range(len(docs))],
)

print(f"Indexed: {collection.count()} documents\n")

# Search by natural language
for q in ["How much does the course cost?", "Can I get a refund?", "Do I need ML experience?"]:
    results = collection.query(query_texts=[q], n_results=2)
    print(f"  Q: {q}")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"    [{dist:.3f}] {doc[:70]}...")
    print()

# Metadata filtering: narrow the candidates BEFORE the vector math
hits = collection.query(query_texts=["course price"], n_results=2,
                        where={"source": "faq_2.txt"})
print(f"filtered: {hits['documents'][0][0][:60]}...")

print("ChromaDB: 5 lines to index, 3 to search. Perfect for prototyping.")

# Output:
#   Indexed: 5 documents
#   (distances: LOWER = closer - the opposite direction of cosine similarity)
#   Q: How much does the course cost?
#     [0.512] GenAI course costs 14999 rupees. Includes Discord, live se...
#     [0.891] Netsetos offers GenAI courses in Hyderabad. 14 modules, 14...
#   filtered: GenAI course costs 14999 rupees. Includes Discord...

- Scene 1: at 8 vectors, checking everything is instant - the query compares against all dots and the nearest glows.

- Scene 2: the space fills with hundreds of thousands of dots; the comparison counter spins and the latency bar blows past budget.

- Scene 3: the index - cluster regions appear; the query checks ~2 percent of vectors; latency snaps back under budget.

- Scene 4: the honest tradeoff - a recall dial: exact scan is 100 percent, ANN is ~95-99, and occasionally a true neighbor sits outside the searched region.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). Every vector database makes this same trade.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: AlloyDB pgvector - SQL + Vectors in One Database

### AlloyDB pgvector - SQL + Vectors in One Database

If you already use PostgreSQL, add vector search without a separate database.

**The question this step answers:** where should vectors live when your app ALREADY has a relational database? Usually: in it. [pgvector](https://github.com/pgvector/pgvector) adds a vector column type and similarity operators to Postgres (`<=>` is cosine distance: lower = closer), so one SQL query can join "nearest chunks" with "WHERE tenant_id = 42 AND has_permission" - no second system, no sync pipeline, transactions included.

> 🏠 **Analogy**
>
> **pgvector is building a home gym instead of joining a fitness club.** Your house (Postgres) already has power, plumbing, and a lock (backups, permissions, transactions). Adding a gym room (the vector extension) means no second membership, no driving between buildings, and your equipment lives next to everything else you own - one address, one key, one bill.
> **Where the analogy breaks down:** a home gym has a ceiling - literally. Past tens of millions of vectors or strict p99 targets, the dedicated club's specialized machines (sharded engines, tuned indexes like ScaNN on AlloyDB or step 4's Vertex) outperform any room you can add to the house.

Vectors live in Chroma; user permissions live in Postgres. A user searches. How many systems answer, and what can go wrong?

**📝 `03_pgvector.py`** - *AlloyDB*

In [ ]:
# ALLOYDB PGVECTOR - SQL + vector search
# pip install asyncpg pgvector sqlalchemy
# Requires AlloyDB instance on GCP

# Schema: documents table with vector column
CREATE_SQL = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS documents (
    id SERIAL PRIMARY KEY,
    content TEXT NOT NULL,
    source VARCHAR(255),
    embedding vector(768),  -- gemini-embedding-001 truncated to 768 (Matryoshka)
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
-- AlloyDB alternative: Google's ScaNN index - faster builds and a
-- smaller memory footprint than HNSW at large row counts.
"""

# Insert with embedding
INSERT_SQL = """
INSERT INTO documents (content, source, embedding) 
VALUES ($1, $2, $3::vector)
"""

# Search by similarity
SEARCH_SQL = """
SELECT content, source, 
       1 - (embedding <=> $1::vector) AS similarity
FROM documents
ORDER BY embedding <=> $1::vector
LIMIT $2
"""

print("AlloyDB pgvector Setup:\n")
print("  1. CREATE EXTENSION vector;")
print("  2. Add vector(768) column to your existing table")
print("  3. CREATE INDEX using hnsw (or ScaNN on AlloyDB)")
print("  4. Search: ORDER BY embedding <=> query_vector\n")
print("  Advantages:")
print("    - Same database for structured data AND vectors")
print("    - SQL joins: combine vector search with filters")
print("    - AlloyDB: GCP-managed, auto-scaling, 100x faster than vanilla PG")
print("    - Example: WHERE category='genai' ORDER BY embedding <=> query")

# Output:
#   AlloyDB pgvector Setup:
#   1. CREATE EXTENSION vector;
#   2. Add vector(768) column to your existing table
#   3. CREATE INDEX using hnsw (or ScaNN on AlloyDB)
#   4. Search: ORDER BY embedding <=> query_vector

---

## 🎯 Concept 4: Vertex AI Vector Search - Billion-Scale Production

### Vertex AI Vector Search - Billion-Scale Production

Google’s managed vector search. Single-digit-ms latency at billion scale (Google-reported). ScaNN.

**What changes at the top tier:** the index and the query service split apart. You build an INDEX from vectors in Cloud Storage (minutes to hours, offline), then DEPLOY it to an ENDPOINT that answers queries. That separation is what buys billion-scale: rebuilds never block serving, shards spread the graph, and Google's ScaNN does the math. The price is ceremony - and an always-on endpoint bill.

> 🚚 **Analogy**
>
> **Vertex Vector Search is a container port, not a courier bag.** The courier bag (Chroma) leaves with you instantly; the port needs berths dug and cranes installed (index build + endpoint deploy) before the first ship docks. But once running, it moves volumes no courier could touch, around the clock, with published schedules (SLAs). You do not build a port for one package - and you do not courier a million containers.
> **Where the analogy breaks down:** ports lock you geographically; managed indexes lock you financially - the endpoint bills BY THE HOUR whether ships arrive or not. An idle port still costs; measure query volume before renting one.

Vertex builds an index in ~40 minutes. Meanwhile production queries keep arriving. Problem?

> 💡 **Info**
>
> ⚠️ This code creates BILLABLE infrastructureThe endpoint below bills by the hour from the moment it deploys, queries or not. Read along first; deploy only with a budget alert set. Delete the endpoint when done.

**📝 `04_vertex_vector_search.py Vertex`** - *AI*

```python
# VERTEX AI VECTOR SEARCH - Billion-scale
# pip install google-cloud-aiplatform
from google.cloud import aiplatform

# Initialize
aiplatform.init(project="your-project", location="us-central1")

# Step 1: Create Index
index = aiplatform.MatchingEngineIndex.create_tree_ah_index(  # MatchingEngine = Vector Search's original API name
    display_name="netsetos-rag-index",
    dimensions=768,             # Gemini embedding dims
    approximate_neighbors_count=10,
    distance_measure_type="COSINE_DISTANCE",
)

# Step 2: Create Endpoint
endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name="netsetos-rag-endpoint",
    public_endpoint_enabled=True,
)

# Step 3: Deploy Index to Endpoint
endpoint.deploy_index(index=index, deployed_index_id="rag_deployed")

# Step 4: Query
# response = endpoint.find_neighbors(
#     deployed_index_id="rag_deployed",
#     queries=[query_embedding],
#     num_neighbors=5,
# )

print("Vertex AI Vector Search:\n")
print("  Scale: billions of vectors")
print("  Latency: single-digit ms even at billion scale (Google-reported)")
print("  Algorithm: ScaNN (Google Research)")
print("  Cost: billed per deployed endpoint-hour + storage - check current rates\n")
print("  When to use:")
print("    ChromaDB:     <1M docs, prototyping, local dev")
print("    pgvector:     1M-10M, need SQL joins, existing Postgres")
print("    Vector Search: 10M+, sub-10ms required, production scale")

# Output:
#   Vertex AI Vector Search:
#   Scale: billions of vectors
#   Latency: single-digit ms even at billion scale (Google-reported)
#   Algorithm: ScaNN (Google Research)
#   When to use:
#     ChromaDB:      under ~1M docs, prototyping, local dev
#     pgvector:      1M-10M, need SQL joins, existing Postgres
#     Vector Search: 10M+, strict latency, production scale
```


**💡 **Which vector store should you use?****

| Feature | ChromaDB | AlloyDB pgvector | Vertex AI VS |
|---|---|---|---|
| Scale | up to ~1M | up to ~10M | 1B+ |
| Setup | pip install | GCP managed | GCP managed |
| SQL support | No | Yes (full SQL) | No |
| Latency @1M | ~50ms | ~20ms | ~5ms |
| Best for | Prototyping | SQL + vectors | Scale & speed |

## Go Deeper: Models, Indexes, and Production Patterns

Steps 5-10 survey the decisions around the build: which model (MTEB honestly read), Matryoshka economics, the wider database ecosystem, what the indexes actually do, distance metrics and caching, and the India-specific model landscape.

---

## 🎯 Concept 5: MTEB Rankings - Commercial vs Open-Source Embedding Models

### MTEB Rankings - Commercial vs Open-Source Embedding Models

Gemini #1 retrieval. Qwen3-8B #1 open-source. Cohere v4 multimodal. Pricing deep dive.

**How to read a leaderboard without being fooled:** [MTEB](https://huggingface.co/spaces/mteb/leaderboard) aggregates dozens of tasks into one score - useful for shortlisting, dangerous for deciding. This step reads the 2026 board honestly: who leads, what the gaps actually mean, and why the two-point difference that headlines announcements rarely survives contact with your corpus.

> 🏆 **Analogy**
>
> **MTEB is the combine, not the season.** Sports scouts watch athletes run drills at the combine - standardized, comparable, public. But teams draft busts and miss stars every year, because drills are not games. A model that aces MTEB's retrieval drills can stumble on YOUR domain vocabulary, YOUR languages, YOUR document lengths. Shortlist at the combine; sign after a tryout on your own field (a labeled-query benchmark from Lesson 8.2 step 3).
> **Where the analogy breaks down:** athletes improve after drafting; a frozen embedding model does not. What you benchmark is what you get - so the tryout matters MORE here than in sports.

**📝 `05_model_tryout.py`** - *language-python*

```python
# The tryout: run YOUR labeled queries against each shortlisted model
# Reuses eval_retrieval from Lesson 8.2 step 3 - same harness, new decision
CANDIDATES = ["gemini-embedding-001", "text-embedding-3-small", "BAAI/bge-m3"]

for model in CANDIDATES:
    embed = make_embedder(model)          # your wrapper per provider
    acc = eval_retrieval(chunks, test_cases, embed_fn=embed)
    print(f"  {model:28s} recall@1 = {acc:.2f}")

# Output: (your corpus decides - MTEB only wrote the shortlist)
#   gemini-embedding-001         recall@1 = 0.92
#   text-embedding-3-small       recall@1 = 0.83
#   BAAI/bge-m3                  recall@1 = 0.88
```


Model A scores 2 points above model B on MTEB. On your Hindi support tickets, who wins?

| Model | MTEB Score | Dims | $/M Tokens | Key Feature |
|---|---|---|---|---|
| Gemini Embedding 001 | 68.32 | 768-3072 | about $0.15 | Best retrieval score, cross-lingual 0.997 |
| Cohere Embed v4 | 65.20 | 256-1536 | about $0.12 | Multimodal (text+image), 128K context |
| Voyage 3.5 | 66.80 | 256-2048 | about $0.06 | Best quality/price, Anthropic recommended |
| OpenAI 3-small | 62.26 | 256-1536 | $0.02 | Cheapest commercial, most deployed |
| Qwen3-Embedding-8B | 70.58 | 32-4096 | about $0.001 | #1 open-source (Apache 2.0), code: 80.68 |
| BGE-M3 | 63.00 | 1024 | about $0.001 | Dense+sparse+ColBERT in ONE forward pass |
| jina-embeddings-v3 | 65.52 | 32-1024 | about $0.001 | 5 task LoRA adapters, ~92% quality at 64 dims |

> ✅ **Info**
>
> 💰 Self-hosting breakeven: ~100M tokens/month
> A single A100 running 24/7 costs about $1,800/month for unlimited tokens at about $0.001/M - **20× cheaper** than the cheapest API. At 100M tokens/month: API costs roughly $2,000 vs self-hosted about $1,800. Above that, self-hosting saves most of the bill. Below 50M tokens/month, APIs are more practical. **Critical:** switching embedding models requires re-embedding your entire corpus - choose carefully and plan migrations.

#### 💡 What just happened

What just happened? The embedding landscape shifts every 3-4 months. **Gemini leads retrieval** (68.32) but has the shortest context (2,048 tokens). **Qwen3-8B dominates open-source** (Apache 2.0, MMTEB #1). **BGE-M3 is uniquely valuable** - dense+sparse+ColBERT in a single forward pass (MIT license). For budget: OpenAI 3-small (about $0.02/M) gives most of the top quality at a fraction of the cost. For multimodal: Cohere v4 is the only option embedding text+images together.

---

## 🎯 Concept 6: Matryoshka Embeddings - Variable Dimensions, Two-Stage Retrieval

### Matryoshka Embeddings - Variable Dimensions, Two-Stage Retrieval

256 dims beats ada-002 (OpenAI's 2022-era default) at 1536. Production storage savings of roughly 75%. The nesting trick.

**The economics of this step:** vector storage and search cost scale with dimensions. Matryoshka-trained models front-load information into the first dimensions, so truncating a 3072-dim vector to 768 - or 256 - keeps most retrieval quality at a fraction of the cost. This is the single cheapest optimization in the whole stack: one parameter (`output_dimensionality`), no re-architecture.

> 🧺 **Analogy**
>
> **Matryoshka embeddings are nesting dolls.** The largest doll holds the full portrait; each smaller doll inside is the SAME face, painted with fewer brushstrokes. Because the artist painted important features first (training objective), even the palm-sized doll is recognizably the same person - you choose the doll that fits your shelf (storage budget), not a different portrait.
> **Where the analogy breaks down:** dolls come pre-nested; ordinary embeddings do NOT truncate gracefully - chop a non-Matryoshka vector and you get noise, not a smaller portrait. The model must be TRAINED for it (gemini-embedding-001 is; check before truncating anything else).

You chop an ordinary (non-Matryoshka) 3072-dim embedding down to its first 256 numbers. What do you get?

- Scene 1: an ordinary vector spreads information evenly - chopping it in half destroys the meaning (similarity craters).

- Scene 2: Matryoshka training front-loads importance - nested outlines at 3072 / 1536 / 768 / 256 dims like Russian dolls.

- Scene 3: truncate to 768 and the document ranking barely moves; at 256 a tiny degradation appears - for 4x-12x less storage.

- Scene 4: two-pass retrieval - shortlist a million dots with 256-dim prefixes, rerank the top-100 with full 3072 - full-precision answers at a fraction of the cost.

Controls: Play/Pause, Reset, speed. output_dimensionality is this exact trick, one parameter away.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> 🎒 How Matryoshka Representation Learning works
> Named after Russian nesting dolls. During training, the loss penalizes poor performance at **every dimension checkpoint** (64, 128, 256, 512, 1024, 3072). This forces the model to front-load the most important semantic information into earlier dimensions. Key difference from naive truncation: standard embeddings distribute information randomly - truncating destroys it. **MRL embeddings are structured so the first N dimensions are a coherent, useful embedding.**
> - **OpenAI 3-large at 256 dims** outperforms ada-002 at full 1536 dims - ~83% storage reduction
> - **Jina v3 retains ~92% quality at just 64 dims** (vs full 1024) - ~94% reduction
> - **Google recommends 768 dims** as the sweet spot: near-peak quality at ~25% of the storage
> - Supported by: OpenAI, Gemini, Cohere v4, Voyage, Jina v3, Qwen3

> 🎒 **Analogy**
>
> **Analogy: JPEG quality slider.** Just as JPEG lets you choose quality vs file size, Matryoshka lets you choose embedding quality vs storage/speed. 256 dims = JPEG at ~80% quality (great for most uses). 3072 dims = JPEG at ~100% (marginally better, 12× larger).

#### 💡 What just happened

What just happened? Matryoshka is the **default production strategy** for cost optimization. The two-stage pattern: shortlist with 256-dim vectors (fast ANN over millions), rerank top-200 with full 3072-dim vectors. At 100M documents: 768 dims = ~400GB vs 3072 dims = ~1.2TB. Same embeddings, same API call - just truncate. **After truncating, re-normalize the vector** - then cosine and dot product agree and either works (truncation changes the norm; normalization restores it).

---

## 🎯 Concept 7: Vector Database Ecosystem - Qdrant, Weaviate, Milvus, LanceDB, Pinecone

### Vector Database Ecosystem - Qdrant, Weaviate, Milvus, LanceDB, Pinecone

Beyond ChromaDB and pgvector. Decision framework by scale and use case.

**Why survey beyond the three we built:** the Google stack covers most needs, but production teams inherit Qdrant clusters, evaluate Pinecone quotes, and read Milvus benchmarks. This step gives you the decision framework - by scale, filtering needs, ops appetite, and lock-in tolerance - so vendor conversations start from YOUR requirements, not their slides.

> 🏙️ **Analogy**
>
> **Choosing a vector database is choosing housing.** Chroma is your studio apartment (cheap, instant, yours). pgvector is the family house you already own with a room converted. Qdrant/Milvus self-hosted are buying a building - control and capacity, but you fix the elevator. Pinecone and managed clouds are the serviced hotel: everything handled, priced per night, and moving out later takes effort.
> **Where the analogy breaks down:** you live in one home; production systems happily use two - an embedded store for dev and a managed cluster in prod, or pgvector for tenant data plus a dedicated engine for the heavy corpus. Mixed estates are normal here.

Your Chroma prototype works great. When is the RIGHT time to migrate to something bigger?

- Scene 1: the prototype - an embedded store inside your app process; 100K vectors, zero infrastructure, instant answers.

- Scene 2: move in with your data - Postgres holds vectors AND relational tables; one SQL query joins a similarity match with a permissions filter.

- Scene 3: the dedicated engine - traffic and corpus grow, Postgres strains, a sharded cluster takes over; latency drops, the ops meter rises.

- Scene 4: the decision flowchart - prototype? embedded. Need SQL joins? pgvector. Past ~50M vectors or strict p99? dedicated. Move when a MEASURED limit bites.

Controls: Play/Pause, Reset, speed. Migrating is code, not magic - start simple.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

| Database | Language | Best For | Scale | Cloud Starter |
|---|---|---|---|---|
| FAISS | C++/Python (in-process library) | Research, custom pipelines | Billions (GPU) | Free (library) |
| ChromaDB | Python | Prototyping, quick starts | up to ~1M | Free (embedded) |
| Qdrant | Rust | Production RAG, filtering | 50M+ | Free 1GB forever |
| Weaviate | Go | Hybrid search, multi-tenant SaaS | 50M+ | about $25/mo |
| Milvus/Zilliz | Go+C++ | Billion-scale, GPU indexes | Billions | Free 5GB |
| LanceDB | Rust | Serverless, S3-backed, edge | Billions (S3) | Free OSS |
| Pinecone | Managed | Zero-ops, fastest time-to-prod | 100M+ | Usage-based |
| Vertex AI Vector Search | Managed (GCP) | Billion-scale, GCP-native (step 4) | 1B+ | Endpoint-hour billing |
| pgvector | PostgreSQL | Already on PostgreSQL, SQL joins | up to ~10M | Free extension |
| Turbopuffer | S3-native | Cost optimization | 3.5T+ docs | about $10/mo |

> ✅ **Info**
>
> 🤑 Key differentiators by database
> - **Qdrant:** Filterable HNSW (filters DURING traversal, ~10% latency cost). Binary quantization (store each dimension in fewer bits, search compressed, rescore survivors at full precision): ~64x memory reduction; vendor benchmarks report ~626 QPS at ~99.5% recall with ~1ms p99.
> - **Weaviate:** Native hybrid search (BM25+vector in one query, alpha blending). Best multi-tenancy: shard-per-tenant with ACTIVE/INACTIVE/OFFLOADED states. SOC II, HIPAA.
> - **Milvus 2.6:** RaBitQ 1-bit quantization (~72% memory reduction, ~4x speed, vendor-reported). GPU indexes (CAGRA). Full-text search 3-4× Elasticsearch throughput. DiskANN for SSD-based billion-scale.
> - **LanceDB:** Zero-copy S3 storage, no server. Used by Midjourney (image search). Multimodal data alongside vectors. about $30M Series A (2025).
> - **Turbopuffer:** S3-native, about $10/month for 1M vectors. 3.5T+ docs in production. Used by Cursor, Notion, Linear.

#### 💡 What just happened

What just happened?**Decision framework:** Prototype → ChromaDB. Production <50M → Qdrant (best filtering, Rust, free tier). Multi-tenant SaaS → Weaviate (native shard-per-tenant). Billion-scale → Milvus (GPU indexes, DiskANN). Serverless/edge → LanceDB (S3-backed, zero server). Zero-ops → Pinecone. Cost-sensitive → Turbopuffer (about $10/mo). Already PostgreSQL → pgvector (<5M). **Migration safety:** build against LangChain/LlamaIndex vector store abstraction - one import change to switch databases.

---

## 🎯 Concept 8: Indexing Algorithms - HNSW, IVF-PQ, ScaNN, DiskANN

### Indexing Algorithms - HNSW, IVF-PQ, ScaNN, DiskANN

Multi-layer graphs. Product quantization. Anisotropic scoring. SSD-based search.

> 📦 **Info**
>
> 📚 Two words, defined once**ANN** = approximate nearest neighbor: instead of comparing your query against every vector (exact, slow), the index checks a small, cleverly-chosen fraction and accepts a tiny chance of missing a true neighbor. **recall@k** = of the k results you should have gotten, how many you actually did. An index at 95% recall@10 returns, on average, 9.5 of the 10 truly-nearest chunks. Every algorithm below is a different bet on the ANN speed-for-recall trade.

**The step where the magic gets explained:** every store so far said "ANN index" and moved on. Here you look inside: how [HNSW](https://github.com/nmslib/hnswlib)'s layered graph finds neighbors in a handful of hops, how IVF-PQ compresses vectors 30x to fit RAM, why Google's ScaNN scores smarter, and how DiskANN serves from SSD. Knowing the mechanics is what turns tuning knobs (M, ef_search, nprobe) from superstition into engineering.

> 🛣️ **Analogy**
>
> **HNSW is a road network: highways, avenues, streets.** To cross a country you do not check every street - you take the highway (sparse top layer, long jumps), exit to avenues (middle layers), then follow streets to the door (dense bottom layer). A few dozen turns replace a million street checks. More roads per intersection (parameter M) means fewer dead ends but a bigger map to store; exploring more routes per trip (ef_search) means better destinations but slower arrival.
> **Where the analogy breaks down:** road maps live on paper; HNSW's whole graph must live in RAM. When the map outgrows memory, you compress the addresses (IVF-PQ quantization) or keep the map on SSD with clever paging (DiskANN) - the tradeoffs the rest of this step tabulates.

An HNSW index answers in about 4ms but recall is 89 percent - you need 95. Rebuild with bigger M, or raise ef_search?

- Scene 1: the layers - dense streets at the bottom, avenues above, a few highways on top.

- Scene 2: the descent - a query takes one or two highway hops, drops a layer, refines, and settles on the nearest neighbors in ~7 hops instead of a million comparisons.

- Scene 3: the knobs - turn M (edges per node) and ef_search (candidates explored) and watch recall and latency bars react.

- Scene 4: where it hurts - the whole graph lives in RAM; at a billion vectors the memory meter overflows, and IVF-PQ or DiskANN take over.

Controls: Play/Pause, Reset, speed. Tune ef_search first - it needs no rebuild.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

| Algorithm | Memory | Recall | Speed | Best For |
|---|---|---|---|---|
| HNSW | High (all in RAM) | 99%+ | 1-2ms | Default <100M vectors |
| IVF-PQ | 4-32× compression | ~90-95% | ~2-10ms | Billions, RAM-constrained |
| ScaNN | Low | ~95%+ | ~1-5ms | GCP (powers Vertex AI) |
| DiskANN | Minimal RAM + SSD | ~95%+ | ~1-3ms | Billion-scale, cost-effective |
| Flat | Full vectors | ~100% (exact) | Slow | <50K vectors (brute-force) |

> 📦 **Info**
>
> ⚙️ Algorithm internals
> - **HNSW:** Multi-layer navigable graph. Upper layers for fast coarse search, layer 0 for precise neighbors. M=16 connections, efConstruction=200 (build), efSearch=100 (query). Memory: N×(M×2×4+8) + N×D×4 bytes.
> - **IVF-PQ:** K-means clusters + product quantization. Split 768-dim vector into 96 sub-vectors, each compressed to 1 byte. 3,072 bytes → 96 bytes (32× compression). Requires training phase. nprobe=32-64 for balance.
> - **ScaNN (Google):** Anisotropic quantization - penalizes error parallel to vector more than orthogonal. 2× better accuracy-speed than standard PQ. Powers Vertex AI Vector Search.
> - **DiskANN (Microsoft):** Vamana graph on NVMe SSD, PQ codes in RAM. about 5,000 QPS at ~95% recall on 1B vectors (Microsoft-reported), single 16-core machine, 64GB RAM. Cost: about $1 per million vectors monthly on SSD vs roughly $10 on RAM.

#### 💡 What just happened

What just happened? HNSW is the **universal default** - used by Qdrant, Weaviate, Milvus, Chroma, pgvector, Pinecone. For <100M vectors with adequate RAM, nothing beats it. **IVF-PQ** for billion-scale when RAM is limited (32× compression). **DiskANN** is the breakthrough: billion-scale on SSD at about $1 per million vectors per month - roughly 10x cheaper than HNSW. **ScaNN** if you're on GCP (powers Vertex AI). Start with HNSW; only switch when scale demands it.

---

## 🎯 Concept 9: Distance Metrics & Production Patterns - Cosine, Dot Product, Hybrid Search, Caching

### Distance Metrics & Production Patterns - Cosine, Dot Product, Hybrid Search, Caching

When metrics are equivalent. Batch processing. Semantic caching (large compute savings).

**The details that bite in production:** three "small" decisions - which distance metric, whether to cache, when to go hybrid - quietly control cost and correctness. The rule for metrics is mechanical (match the model card; on normalized vectors cosine and dot product agree). Caching and hybrid search are where real money and quality hide.

> ⚖️ **Analogy**
>
> **Distance metrics are measuring tape vs bathroom scale.** Both measure "size", but you cannot swap them mid-project: a cabinet specced in centimeters fails when checked in kilograms. The embedding model was TRAINED against one metric - its model card says which tape it used; measure with the same one. On normalized vectors the tapes happen to agree (cosine = dot product), which is why many stores just normalize everything.
> **Where the analogy breaks down:** a wrong tape fails visibly; a wrong metric fails SILENTLY - retrieval still returns results, just subtly worse-ranked ones. Nothing errors. That silence is why this boring step exists.

**📝 `09_metric_check.py`** - *language-python*

In [ ]:
# Prove it in six lines: on NORMALIZED vectors, cosine == dot product
import numpy as np

a, b = np.random.rand(768), np.random.rand(768)
an, bn = a / np.linalg.norm(a), b / np.linalg.norm(b)   # normalize once at index time

cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"cosine(a,b)      = {cosine:.6f}")
print(f"dot(a_norm,b_norm)= {np.dot(an, bn):.6f}")  # identical - and cheaper per query

# Output:
#   cosine(a,b)      = 0.751592
#   dot(a_norm,b_norm)= 0.751592

A teammate switches similarity from cosine to dot product "for speed" on UNNORMALIZED vectors. What happens?

> ✅ **Info**
>
> 📏 Distance metrics - when they're equivalent
> **For L2-normalized vectors (most embedding models output these):** cosine similarity = dot product, and L2 distance = 2×cosine_distance. All three produce **equivalent rankings**. Use dot product for speed (no normalization step). For **Matryoshka truncated vectors:** re-normalize after truncating, then cosine and dot product agree - the safe default that avoids any norm surprise. Historically some pipelines just used dot product to avoid the re-normalization step, but re-normalizing is clearer and loses nothing. Cosine re-normalizes and loses the magnitude signal. Always verify your model normalizes outputs. OpenAI, Cohere, sentence-transformers: yes. Some custom models: no.

> 📦 **Info**
>
> 🚀 Production embedding patterns
> - **Batch processing:** OpenAI Batch API = ~50% cost reduction (24h window). Sequence-length sorting before batching reduces padding waste ~20-40%. Mixed-precision (FP16) halves memory.
> - **Multi-layer caching:** L1 in-memory LRU (~1ms) → L2 Redis (~5ms, shared) → L3 persistent (~10-50ms). Key: md5(content).
> - **Semantic caching:** Cache results for similar queries (similarity >0.85). **80-95% compute savings**, p95 from 2.1s to 450ms.
> - **Hybrid search:** dense embeddings catch MEANING; BM25 (classic keyword scoring) catches EXACT terms like error codes and product IDs; RRF (Reciprocal Rank Fusion) merges the two ranked lists by position. Together: ~26-31% NDCG uplift (a ranking-quality score) in published evaluations. Alpha≈0.3 for technical docs, α≈0.7 for natural language. Native in Weaviate, Qdrant, Milvus 2.6.
> - **Embedding versioning:** Blue-green pattern. Generate new embeddings alongside old. Parallel evaluation. Atomic switch via is_current flag. Track model_name, version, source_hash.

#### 💡 What just happened

What just happened? Three production patterns matter most: **Matryoshka two-stage retrieval** (shortlist with small dims, rerank with full - saves ~75% storage), **semantic caching** (80-95% compute savings for repeat/similar queries), and **hybrid search** (+26-31% NDCG, non-negotiable for production). Build embedding versioning from day one - model migrations are inevitable.

---

## 🎯 Concept 10: India Embeddings - Indic Models, Cross-Lingual Gaps, AWS Mumbai, DPDP

### India Embeddings - Indic Models, Cross-Lingual Gaps, AWS Mumbai, DPDP

bge-multilingual-gemma2 ~35% over BGE-M3 (Sarvam-reported). Tokenizer fertility. Self-hosting on Mumbai.

**Why India needs its own embedding step:** the English leaderboard hides three India-specific realities - Indic retrieval quality varies wildly across models, cross-lingual queries (Hindi question, English documents) fall into gaps single models cannot bridge, and DPDP data-residency rules can force self-hosting decisions that change the entire model shortlist.

> 🌎 **Analogy**
>
> **A multilingual embedding model is an interpreter, not a native speaker.** A good interpreter (BGE-M3, multilingual Gemini) handles Hindi-to-English meetings competently - but idioms, code-switching ("mera refund kab aayega?"), and legal vocabulary strain them in ways monolingual English work never reveals. You audition interpreters ON the meetings you actually hold: mixed Hindi-English support tickets, not TOEFL essays.
> **Where the analogy breaks down:** one interpreter per meeting is enough; production Indic retrieval often runs TWO passes - dense multilingual embedding on the original query PLUS keyword search on a translation, fused with RRF - because each catches what the other misses.

**📝 `10_dual_retrieval.py`** - *language-python*

```python
# Dual retrieval for cross-lingual queries: dense (Hindi) + BM25 (English) + RRF
def rrf_merge(rank_lists, k=60):
    """Reciprocal Rank Fusion - reward docs ranked high by EITHER retriever."""
    scores = {}
    for ranks in rank_lists:
        for pos, doc_id in enumerate(ranks):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)

dense_ranks = dense_search(hindi_query)             # BGE-M3 / gemini multilingual
bm25_ranks  = bm25_search(translate(hindi_query))   # keyword pass on English
final = rrf_merge([dense_ranks, bm25_ranks])
print(final[:3])

# Output:
#   ['refund_policy_en', 'refund_faq_hi', 'pricing_en']
```


A Hindi query needs English documents. One multilingual dense pass misses relevant docs. Your move?

> 📦 **Info**
>
> 🇮🇳 India-specific embedding patterns
> - **Best model:** bge-multilingual-gemma2 (Sarvam AI's choice) - about 35% better recall than BGE-M3 for Indian languages (Sarvam-reported). Trade-off: 3× larger, mitigated by scalar 8-bit quantization. BGE-M3 as budget alternative (MIT, 568M params).
> - **Cross-lingual gap:** Hindi→English retrieval achieves roughly 60-80% of monolingual accuracy. Mitigation: translate query to English + dual retrieval (dense on original + BM25 on English) + RRF merge + multilingual reranker (bge-reranker-v2-m3).
> - **Tokenizer fertility (tokens produced per word - higher means more expensive):** standard BPE over-segments Devanagari roughly 3-5x vs English. Sarvam tokenizer: 1.4-2.1 fertility. Same Hindi doc costs 3-5× more to embed with poor tokenizer. Factor into cost planning.
> - **Self-hosting (AWS Mumbai):** r6g.xlarge (32GB Graviton2) at about $94/month. Qdrant single node handles 10M vectors. Total: about $150/month. ~150-200ms latency savings vs US regions.
> - **DPDP Act:** Embeddings = derived personal data. RBI mandates payment data in India. SEBI mandates governance data locally. Penalties: ₹250 crore (about $30M). Safe default: keep vector stores, GPU, and storage in Indian regions.
> - **Romanized Hindi:** bhasha-embed-v0 - first model supporting Romanized Hindi + Devanagari + English cross-lingual alignment. Critical for India's 250M code-mixed users.

#### 💡 What just happened

What just happened? Indian RAG has three cost traps: **tokenizer fertility** (3-5× more expensive with standard tokenizers), **cross-lingual accuracy gaps** (~20-40% lower), and **DPDP compliance risk** (embeddings are derived personal data). The production stack: bge-multilingual-gemma2 (quality) or BGE-M3 (budget) → Qdrant/Milvus on AWS Mumbai (about $150/month) → dual retrieval for cross-lingual → ChrF evaluation (a character-level quality score, robust for Indic scripts). Total infra: about $300-600/month (the ~$150 vector node plus embedding serving, monitoring, and backups) for a modest production system.

> 📦 **Info**
>
> 🔗 Where this goes next (Module 8)
> - **In Lesson 8.4 (Advanced Retrieval)**, these stores become the substrate: hybrid search, reranking, and query transforms all run ON TOP of the indexes you tuned today - and we'll come back to RRF fusion in full code.
> - **In Lesson 8.5 (RAG with LangChain)** and **8.6 (LlamaIndex)**, the vector-store abstraction lets you swap Chroma for Qdrant for Vertex behind one interface - the migration exercise below previews it.
> - **In Lesson 8.11 (RAG Evaluation & RAGAS)**, recall@k grows into a full evaluation harness - the tryout discipline from step 5 becomes a CI gate.

## Key Takeaways

> 📦 **Info**
>
> - **Embeddings are a managed-API decision now.** gemini-embedding-001 (Matryoshka, task_type) is the course default; open-source (Qwen3, BGE-M3) wins when data cannot leave or scale makes APIs expensive.
> - **Vector storage is a scaling journey, not a product pick.** Embedded library (Chroma) to prototype, pgvector when vectors must join relational data, dedicated engine (Vertex, Qdrant, Milvus) when a measured limit bites.
> - **Every index is a recall-for-speed trade.** HNSW pays in RAM, IVF-PQ pays in recall, DiskANN pays in latency - know which currency you are spending.
> - **Matryoshka truncation is free money.** Shortlist with small prefixes, rerank with full vectors - most of the quality at a fraction of the cost.
> - **Match the distance metric to the model card,** filter with metadata BEFORE the vector search, and benchmark on YOUR corpus - leaderboard rank is not your recall.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each exercises a decision you will make in production.

---

## 🎓 Summary

You've completed the practical part of **Embeddings & Vector Stores**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.3.html` - regenerate this notebook whenever the source HTML is updated.*
